# A/B тестирование по части №1

## Результаты EDA-этапа

- Датасет очищен: нет дубликатов по user_id;
- Проведена проверка на сопостовимость групп: в treatment и control примерно одинаковое число записей (различие на 0.06%);
- Временные границы эксперемента совпадают для обеих групп;
- Поток пользователей равномерен - сильно различимые изменения только в первый и последний день.

**Итого имеем обработанный датасет с 287706 записями**

----

В рамках целевой метрики будем измерять конверсию в целевое действие по признаку* *'converted'*, где:
- '0' пользователь не перешел дальше по конверсионной воронке с лендинга;
- '1' пользователь прошел по воронке дальше.

Конверсия для каждой группы рассчитывается как: conversition = (кол-во людей совершивших целевое действие)/(общее число людей в данной группе)

**Метрика для A/B-тестирования: конверсия (CR)**

*Важно уточнить, что данный показатель и его интерпритация - допущение от команды, основанное на базе предоставнного кейса.

## Загрузим очищенный датасет

In [57]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [58]:
df = pd.read_csv('clean_data_part_1.csv')
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,user_id,timestamp,group,landing_page,converted
0,922696,2025-01-02 13:42:05.378582,treatment,new_page,0
1,781507,2025-01-02 13:42:15.234051,control,old_page,0
2,737319,2025-01-02 13:42:21.786186,control,old_page,0
3,818377,2025-01-02 13:42:26.640581,treatment,new_page,0
4,725857,2025-01-02 13:42:27.851110,treatment,new_page,0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287706 entries, 0 to 287705
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       287706 non-null  int64 
 1   timestamp     287706 non-null  object
 2   group         287706 non-null  object
 3   landing_page  287706 non-null  object
 4   converted     287706 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.0+ MB


## Гипотезы

Сформулируем нулевую гипотезу (H_0), альтернативную гипотезу (H_1) и уровень значимости (a)

**H_0** - новая версия лендинга **не влияет** на конверсию пользователь в целевое действие (CR)

Формализовано: СR_control = CR_treatment (со стат. погрешностью)

**H_1** - новая версия лендинга **влияет** на конверсию пользователь в целевое действие (CR). Фактически влияение может быть как в лучшую, так и в худшую сторону - двухстороняя гипотза.

Формализовано: СR_control != CR_treatment (со стат. погрешностью)

**Уровень значимости (a)**

Для бейзлайна установим стандартное значение a = 0.05.
В рамках указанного теста (двухстороннего) будет соответственно 0.05/2 = 0.025 с каждой стороны.

## Выбор статистического теста

**Командой были выбраны 2 теста:**
- Z-test
- Chi-test (для проверки результатов Z-теста)

Причины выбора Z-test (для пропорциии, то применимо для CR):
1) Применяется при > 30 наблюдениях (выполнено - в каждой из категорий > 140k). Для теста с пропорцией данный критерий заменяется требованием, чтобы в каждой группе было >= 5 наблюдейний (аналогично выполнено).
2) Известно стандартное отклонение генеральной совокупности - это требование для классического z-теста. Для пропорции стандартное отклонение можно вычислить, поскольку работаем с бинарной (Бернулиевской величиной).
3) Классический z-test работает с непрерывным распределением, а пропорционный может работать с дискретными данными.
4) Должно быть нормальное распределение статистик - данное правило автоматически выполняется для бинарной величины при выополненном пункте 2 (см. ссылки ниже) - вытекает из ЦПТ.
5) Случайность выборки - допущение, что пользователи случайно назначались в группу (control/treatment).
6) Независимость наблюдейний:
- каждый пользователь присутствует только 1 раз в датасете (результат EDA);
- группы не пересекаются.

Теоретическая база из следующих источников:
- https://habr.com/ru/companies/otus/articles/805961/ (про выбор стат. теста)
- https://sky.pro/wiki/analytics/statisticheskij-test-osnovnye-vidy-primenenie-i-analiz-rezultatov/  (про выбор стат. теста)
- https://practicum.yandex.ru/blog/z-test-proverka-gipotez/  (z-тест) 
- https://en.wikipedia.org/wiki/Two-proportion_Z-test  (z-тест для пропорций)
- https://mindthegraph.com/blog/ru/chi-square-test/  (хи-тест)

**Проверим пункт 1 (про 5 и более наблюдейний в группе)**

Для проверки используем корреляционный тест Пирсона: https://www.geeksforgeeks.org/python/python-pearsons-chi-square-test/

In [60]:
from scipy.stats import chi2_contingency

In [61]:
valid_crosstab = pd.crosstab(df['group'], df['converted'])
valid_crosstab

converted,0,1
group,,
control,126509,17309
treatment,126807,17081


In [62]:
chi_2, p, dof, expec = chi2_contingency(valid_crosstab)
expec

array([[126627.18361105,  17190.81638895],
       [126688.81638895,  17199.18361105]])

**Вывод: теоретическое кол-во записей превышает 5 (даже близко к реальным), следовательно тест применим**

## Z-тест

Проведем A/B-тестирование, метрика - CR

Посчитаем конверсию в контрольной группе: CR_control

In [63]:
df_control = df[df['group'] == 'control']

all_control_amount = len(df_control)
converted_in_control = df_control['converted'].sum()

CR_control = converted_in_control/all_control_amount

print(f'Цифры в control: {converted_in_control} --- {all_control_amount} ---- {np.round(CR_control, 2)*100}%')
print(CR_control)

Цифры в control: 17309 --- 143818 ---- 12.0%
0.12035350234323937


In [64]:
df_treatment = df[df['group'] == 'treatment']

all_treatment_amount = len(df_treatment)
converted_in_treatment = df_treatment['converted'].sum()

CR_treatment = converted_in_treatment/all_treatment_amount

print(f'Цифры в treatment: {converted_in_treatment} --- {all_treatment_amount} ---- {np.round(CR_treatment, 2)*100}%')
print(CR_treatment)

Цифры в treatment: 17081 --- 143888 ---- 12.0%
0.11871038585566551


**Разница в конверсиях**

Абсолютная разница и относительная

In [65]:
print(CR_control - CR_treatment)

0.0016431164875738563


In [66]:
print(f'{((CR_control/CR_treatment) - 1) * 100}%')

1.3841387808912087%


**Z-Тест**

Создадим матрицу сопряженности своими руками черезе np.array, чтобы в дальнейшем передать ее в метод: proportions_ztest

- https://www.geeksforgeeks.org/machine-learning/how-to-perform-a-one-proportion-z-test-in-python/

In [67]:
from statsmodels.stats.proportion import proportions_ztest

In [68]:
conversitions_values = np.array([converted_in_control, converted_in_treatment])
all_values = np.array([all_control_amount, all_treatment_amount])

a = 0.05

z_statistica, p_value = proportions_ztest(
    conversitions_values,
    all_values,
    alternative = 'two-sided'
)

print(f'Z-статистика = {z_statistica}')
print(f'p-value = {p_value}')

Z-статистика = 1.358358250077283
p-value = 0.17435003516629866


In [69]:
if p_value < a:
    print('Отвергаем H_0')
else:
    print('Не отвергаем (оставляем) H_0')

Не отвергаем (оставляем) H_0


**Проверим результаты через Хи-тест (хи-квадрат)**

- https://stats.stackexchange.com/questions/173415/at-what-level-is-a-chi2-test-mathematically-identical-to-a-z-test-of-propo

Для корректности сравнения чисел с плавающей точек будем использовать метод np.isclose, поскольку == работает неточно, в силу окргулений.

- https://www.geeksforgeeks.org/python/python-filter-out-integers-from-float-numpy-array/?ysclid=mpiwgtwdzs566660158#:~:text=using%20NumPy%20functions.-,Using%20np.isclose(),-This%20method%20compares

In [71]:
chi_2, p, dof, expec = chi2_contingency(valid_crosstab, correction=False)

print(chi_2, np.isclose(chi_2, z_statistica**2))
print(p, np.isclose(p, p_value))

print(dof)

1.8451371355529882 True
0.17435003516629843 True
1


**Вывод:** не отвергаем нулевую гипотезу, следовательно новая версия лендинга **не влияет** на конверсию.

Подробнее:
- статистически значимых различий между группами не обнаружено;
- новая версия не изменяет поведение пользователей в пределах статической погрешности (5%)


Также отметим: **тест проведен корректно**:
- z-тест^2 совпал с хи-квадрат;
- p_value совпал в обоих случаях.

## Анализ CR на доверительном интервале

Поскольку по результатам A/B-теста была сохранена H_0, следовательно существует 2 варианта:
1) эффект от изменений на странице статически мал;
2) эффект есть, но не смогли его обнаружить (например, из-за мощности теста)

Для оценки используем соответствующий модуль для построения доваерительного интервала (для пропорций): https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.confint_proportions_2indep 

In [74]:
from statsmodels.stats.proportion import confint_proportions_2indep

low_line, high_line = confint_proportions_2indep(
    converted_in_treatment,
    all_treatment_amount,
    converted_in_control,
    all_control_amount,
    method = 'newcomb',
    alpha = 0.05
)

print(low_line, high_line)

-0.004014023932366113 0.0007277571790263106


**Вывод:** с доверительным интервалом 95% истинная разница конверсий (экспериментальной и контрольной групп, т.е. CR_treatment - CR_control)
находится в диапазоне: [-0.0040, 0.0007]. 

Даже если изменения в метрике и есть, то новая версия лендинга изменяет (не факт что увеличивает CR) на 0.07% - довольно малый эффект.